# 6 · Resilience & Persistence

Make the system production-ready:

- **Retries & fallback** — survive flaky tools/models (`ToolRetryMiddleware`, `ModelFallbackMiddleware`).
- **Durable threads** — each ticket persists in `aurora_checkpoints_db` and can resume later.

In [1]:
%run aurora_common.py

C:\Users\akhawaja\git\cs4603\wk4_capstone\aurora_common.py:42: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Retrying a flaky tool

In [2]:
from langchain.agents.middleware import ToolRetryMiddleware

calls = {"n": 0}


@tool
def lookup_inventory(product_id: int) -> str:
    """Check inventory (simulated flaky API)."""
    calls["n"] += 1
    if calls["n"] < 3:
        raise RuntimeError(f"transient error (attempt {calls['n']})")
    return f"product {product_id}: 42 in stock"


retry_agent = create_agent(
    model=llm_noreason,
    tools=[lookup_inventory],
    middleware=[ToolRetryMiddleware(max_retries=3, initial_delay=0.0, backoff_factor=1.0, jitter=False)],
)

calls["n"] = 0
r = retry_agent.invoke({"messages": [HumanMessage(content="How many units of product 10 are in stock?")]})
r["messages"][-1].pretty_print()
print("tool attempts:", calls["n"])

================================== Ai Message ==================================

There are 42 units of product 10 in stock.
tool attempts: 3


## Durable ticket threads (Postgres checkpointer)
The same `thread_id` resumes a ticket with full memory. State lives in `aurora_checkpoints_db`.

In [3]:
agent = create_agent(
    model=llm_noreason,
    tools=[],
    system_prompt="You are a support assistant.",
    checkpointer=create_pg_checkpointer(),
)

ticket_cfg = make_thread_config("ticket-demo-001")

agent.invoke({"messages": [HumanMessage(content="Hi, my name is Ada and my order is 1001.")]}, ticket_cfg)
r = agent.invoke({"messages": [HumanMessage(content="What order number did I just mention?")]}, ticket_cfg)
r["messages"][-1].pretty_print()

# Inspect persisted state
state = agent.get_state(ticket_cfg)
print("\nMessages persisted for this ticket:", len(state.values["messages"]))

  • database 'aurora_checkpoints_db' already exists


================================== Ai Message ==================================

You mentioned order number **1001**.

Messages persisted for this ticket: 4


### Definition of done
- Flaky tool succeeds after retries.
- A ticket thread remembers earlier turns across separate `invoke` calls.
- Bonus: add `ModelFallbackMiddleware` with a backup model.